# Variational Inference: ELBO, Mean Field, and Optimization

> Advanced Machine Learning for Artificial Intelligence · Dongduk Women's University · Professor Wonsang You · 2026-2  
> Week 4 · Tuesday, September 22, 2026 · 120-minute lesson

**Learning outcomes.** Derive the ELBO identity; distinguish model and variational parameters; derive and implement mean-field coordinate updates; compare deterministic and stochastic gradients; diagnose approximation versus optimization error; assemble an interactive inference explorer.

**Reading.** Murphy, *Probabilistic Machine Learning: Advanced Topics*, online edition dated December 10, 2025: `10.1.1–10.1.2 (printed pp. 439–442), `10.2.1 (pp. 445–451), `10.3.1–10.3.2 (pp. 455–459); `10.4.1 (p. 471) for the stochastic-gradient bridge. These are printed page numbers; the locally supplied PDF places them 34 pages later. The lesson develops original examples and derivations rather than reproducing textbook pages.

**Prerequisites.** Bayes' rule, multivariate Gaussian mean/covariance/precision, expectations, KL divergence, partial derivatives, and W2 graphical-model factorization.

CPU runtime; save a copy and run from top to bottom. Unfinished exercises may be skipped. The demo uses original synthetic data, no uploads.

# W4 shared inference API

All logarithms are natural; two-dimensional arrays follow the model in the lecture.

## [GaussianModel](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L14)

Exact reference for one observed vector.

x and mean have shape (2,); noise, precision and covariance have shape
(2,2); log_evidence is a scalar log density, not a probability.
Construct with gaussian_model; treat stored arrays as read-only.

## [gaussian_model](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L36)

Return an exact posterior/evidence reference, without fitting or sampling.

rho: observation-noise correlation, strictly between -1 and 1; it is NOT
posterior correlation. x: observed length-2 vector, default (1,-1).
Returned arrays are new. Raises ValueError for invalid dimensions/numbers.
Example: gaussian_model(.8).mean is approximately [.833333,-.833333].

## [mean_field_optimum](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L63)

Return the reverse-KL-optimal diagonal Gaussian (mean, variance).

Both outputs have shape (2,). The mean is the exact posterior mean;
variance[j] = 1/precision[j,j], generally NOT covariance[j,j].
Inputs are unchanged; no iteration or download occurs.

## [elbo](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L73)

Compute E_q[log p(z)+log p(x|z)] + H(q), directly.

mean and variance are length-2 vectors for q=N(mean,diag(variance)).
Output is a scalar in nats; variance means sigma squared, not sigma.
Includes every normalization constant; does NOT subtract KL from the
evidence oracle internally. Inputs are not mutated.

## [diagonal_kl](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L91)

Return KL(q || exact posterior) in nats using the Gaussian KL formula.

mean/variance: length-2 vectors, positive variance. This independently
checks log_evidence - elbo; it is not a sample estimate or a symmetric
distance. Inputs unchanged. A tiny negative roundoff near zero is possible.

## [elbo_gradient](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L106)

Analytic ELBO gradients with respect to mean and log standard deviation.

Inputs are length-2 vectors. variance=exp(2*log_std). Returns two length-2
arrays (dL/dmean, dL/dlog_std), for gradient ASCENT. No state is changed.
Using log standard deviation keeps variances positive.

## [reparameterized_gradient](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L119)

Estimate the same gradients by z=mean+exp(log_std)*epsilon.

samples: positive integer; seed initializes a local NumPy generator.
Analytic Gaussian entropy supplies the +1 in the log-std gradient.
Returns two (2,) sample-average gradients. No global random state,
downloads or fitting. Estimates fluctuate; no per-step ascent guarantee.

## [run_vi](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L139)

Run sequential Gaussian CAVI or analytic ELBO gradient ascent.

method is 'cavi' or 'gradient'; steps is a nonnegative integer. CAVI
updates coordinate 0 then 1 using the newest values; a recorded step is
one complete sweep. Gradient updates mean and log_std jointly with an
Armijo backtracking line search from learning_rate (>0).
Returns means/variances of shape (steps+1,2), elbos (steps+1,),
step_sizes (steps,); NaN for CAVI, and method. Row 0 is initialization.
Inputs/model are not mutated. Raises on invalid inputs or line-search
failure; never replaces a failed calculation with a fabricated result.

## [diagnostics](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L197)

Return scalar/JSON-ready diagnostics for one diagonal approximation.

Separates approximation gap at the optimal diagonal q from optimization
gap above that optimum. Gaps are KL quantities in nats. Only a numerical
optimization gap within 1e-10 of zero is rounded to zero for display.

## [trace_figure](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L219)

Return a Matplotlib Figure: covariance geometry and ELBO trace.

Contours have unit Mahalanobis radius (not 68% joint credible regions).
trace is a run_vi result. Creates no files or server; caller may savefig.

## [vi_view](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L252)

App callback: settings -> (Figure, JSON-ready diagnostics).

rho controls observation noise; method is cavi/gradient; steps counts full
sweeps/steps. Each call constructs a fresh model and trace; no retained fit.
learning_rate applies only to gradient ascent. Default observed x=(1,-1).

## [build_vi_app](https://github.com/lunalab-ai/advML/blob/2026-fall-w3/src/advml_vi.py#L269)

Return an unlaunched Gradio Blocks app using vi_view.

Requires Gradio 6.27.0. No server/download starts here. Call launch with
share=True in Colab for a temporary link tied to the running runtime.
Students extend the displayed diagnostics in their own exercise callback.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, sys, tempfile, urllib.request, subprocess, os
os.environ["GRADIO_ANALYTICS_ENABLED"]="False"
for package, version in [("numpy","2.2.6"),("matplotlib","3.10.3"),("gradio","6.27.0")]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet",f"{package}=={version}"])
CODE_REF="2026-fall-w3"
EXPECTED_SHA256="699c3e5f7f75bee9dfdceff05e4956f7bd2bf0b9abff6c0eacf11a9ac3f1ad2a"
CODE_URL=f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/advml_vi.py"
local=Path("src/advml_vi.py")
if local.is_file() and hashlib.sha256(local.read_bytes()).hexdigest()==EXPECTED_SHA256:
    module_path=local.resolve()
else:
    module_path=Path(tempfile.gettempdir())/f"advml_vi_{EXPECTED_SHA256[:16]}.py"
    if not module_path.is_file() or hashlib.sha256(module_path.read_bytes()).hexdigest()!=EXPECTED_SHA256:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(CODE_URL,timeout=30) as response:
                    payload=response.read()
                if hashlib.sha256(payload).hexdigest()!=EXPECTED_SHA256:
                    raise RuntimeError("Course code checksum mismatch; stop and report, do not bypass.")
                module_path.write_bytes(payload)
                break
            except Exception as exc:
                if attempt==2:
                    raise RuntimeError("Fixed course code could not be prepared. Check connection and rerun; a complete public checkout is the offline alternative.") from exc
spec=importlib.util.spec_from_file_location("advml_vi",module_path)
vi=importlib.util.module_from_spec(spec);sys.modules["advml_vi"]=vi;spec.loader.exec_module(vi)
import numpy as np, matplotlib, gradio as gr
from IPython.display import display
IN_COLAB="google.colab" in sys.modules
print({"Python":sys.version.split()[0],"NumPy":np.__version__,"Matplotlib":matplotlib.__version__,"Gradio":gr.__version__,"code":CODE_REF,"sha256":EXPECTED_SHA256})


## Predict, then inspect the exact reference
Noise correlation is a model input; posterior correlation is an output. Do you expect them to be equal?


In [ ]:
model=vi.gaussian_model(.8)
trace=vi.run_vi(model,steps=10)
display(vi.trace_figure(model,trace))
print(vi.diagnostics(model,trace["means"][-1],trace["variances"][-1]))


## E1 · Check the identity

At m=(0,0), v=(1,1), fill the gap and compare it to independently computed reverse KL. State its units.


In [ ]:
e1_gap=None  # evidence minus direct ELBO


## E2 · Which variance?

Compute the best diagonal-family variance and the target marginal variance. Explain why they differ.


In [ ]:
e2_variance=None
# Compare your result with np.diag(model.covariance).


## E3 · One sequential sweep

Starting from m=(2,-2), compute coordinate 0 then coordinate 1 using the new value. Keep a copy of the initial vector.


In [ ]:
e3_mean=None


## E4 · Debug an optimizer

A proposed update is m_new=m−0.01*gradient. Repair the sign for ELBO maximization and verify improvement with v held fixed.


In [ ]:
e4_new_mean=None


## E5 · Sampling error

Estimate gradient RMSE over 20 independent seeds for N=16,256,4096 at m=(0,0), a=(0,0). Does every individual realization have to improve?


In [ ]:
e5_errors=None


## E6 · Diagnose the remaining gap

Compare 0 and 30 CAVI sweeps at rho=0 and .8. Classify each reported gap and propose a useful intervention.


In [ ]:
e6_table=None


## E7 · Build your own diagnostic app

Write a callback taking rho and steps. Return a plot and JSON with an added variance_retention list: q variance divided by exact marginal variance. Connect it to two Gradio inputs and two outputs. Test rho=0 and .8 before launching.


In [ ]:
def my_view(rho,steps):
    # TODO: call vi.vi_view, add your diagnostic, return Figure and dictionary.
    return None, {"status":"Complete E7 before launching your own app."}
my_app=None  # construct your own interface after completing my_view


## Completed demonstration
Run this even if exercises are unfinished. Change noise correlation and method; explain which gap changed. In Colab the sharing URL is temporary and expires with the runtime. The notebook is the permanent entry point.


In [ ]:
demo=vi.build_vi_app()
if IN_COLAB:
    demo.launch(share=True,prevent_thread_lock=True)
else:
    print("App constructed. For a local browser demo call demo.launch() manually.")


In [ ]:
check_model=vi.gaussian_model(.8)
for method in ["cavi","gradient"]:
    checked=vi.run_vi(check_model,method,60)
    assert checked["means"].shape==(61,2)
    assert np.all(np.diff(checked["elbos"])>=-1e-9)
    for m,v,b in zip(checked["means"],checked["variances"],checked["elbos"]):
        assert np.isclose(check_model.log_evidence-b,vi.diagonal_kl(check_model,m,v))
        assert b<=check_model.log_evidence+1e-10
for rho in [0.,.8]:
    fig,metrics=vi.vi_view(rho,"cavi",30)
    assert len(fig.axes)==2 and metrics["optimization_gap"]<1e-8
assert vi.vi_view(0.,"cavi",30)[1]["approximation_gap"]<1e-10
assert vi.vi_view(.8,"cavi",30)[1]["approximation_gap"]>.2
print("W4 runtime checks passed: independent identity, ascent, shapes, two app settings.")


## Checkpoint quiz

1. Why can ELBO maximization avoid evaluating the evidence during fixed-model inference?

2. Does mean field assert posterior independence?

3. Why is the optimal Gaussian mean-field variance 1/Pjj rather than Cjj?

4. What is guaranteed by an exact sequential CAVI update?

5. Why does the log-standard-deviation gradient contain a +1?

6. Can increasing iterations eliminate every ELBO gap?

7. What distinguishes the W2 Ising variational objective from a log-evidence bound?
